Format Dataset for Supervised Fine-Tuning

In this notebook, I will convert the Human Message Rewriter dataset into a supervised fine-tuning format.
Each example currently has:
- instruction
- input
- output

For fine-tuning, we will format each example as a conversation:
- system message: defines the assistant's behavior
- user message: gives the rewriting instruction and input text
- assistant message: provides the target rewritten output

This formatted dataset will later be used to fine-tune the model.

# Importing libraries

In [1]:
import json
from pathlib import Path
import pandas as pd

# Configuration Setup

## Setup Paths

In [2]:
train_path = Path("../data/processed/train.jsonl")
validation_path = Path("../data/processed/validation.jsonl")
test_path = Path("../data/processed/test.jsonl")

print(f"Train file Path : {train_path} | Exists: {train_path.exists()}")
print(f"Validation file Path : {validation_path} | Exists: {validation_path.exists()}")
print(f"Test file Path : {test_path} | Exists: {test_path.exists()}")


Train file Path : ..\data\processed\train.jsonl | Exists: True
Validation file Path : ..\data\processed\validation.jsonl | Exists: True
Test file Path : ..\data\processed\test.jsonl | Exists: True


## load train. validation and test splits

In [3]:
def load_jsonl(file_path):

    rows = []

    with file_path.open("r", encoding = "utf-8") as file:
        for line in file:
            line = line.strip()

            if line:
                rows.append(json.loads(line))

    return rows

In [5]:
train_rows = load_jsonl(train_path)
validation_rows = load_jsonl(validation_path)
test_rows = load_jsonl(test_path)

print(f"Test rows:",len(train_rows))
print(f"Validation rows:",len(validation_rows))
print(f"Test rows:",len(test_rows))

Test rows: 240
Validation rows: 30
Test rows: 30


# Format dataset

## Test on one cell

In [6]:
example = train_rows[0]
print("Example row from train set:", example)

Example row from train set: {'id': 'ex_0003', 'category': 'workplace', 'instruction': 'Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.', 'input': 'Please be informed that I have attached the required document for your reference.', 'output': 'I’ve attached the required document for your reference.', 'source': 'synthetic_curated'}


In [7]:
print("Instruction:")
print(example["instruction"])

print("\nInput:")
print(example["input"])

print("\nOutput:")
print(example["output"])

Instruction:
Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.

Input:
Please be informed that I have attached the required document for your reference.

Output:
I’ve attached the required document for your reference.


## Setup the system message

In [8]:
SYSTEM_MESSAGE = (
    "You are a helpful writing assistant. "
    "Rewrite messages so they sound natural, clear, and human while preserving the original meaning. "
    "Do not add new information."
)

SYSTEM_MESSAGE

'You are a helpful writing assistant. Rewrite messages so they sound natural, clear, and human while preserving the original meaning. Do not add new information.'

## Format as message function

In [9]:
def format_as_message(row):
    user_message = (
    f"Instruction: {row['instruction']}\n\n"
    f"Input: {row['input']}\n\n"
    )

    assistant_message = row["output"]

    return {
        "id": row["id"],
        "category": row["category"],
        "messages": [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]
    }

## Single conversation example

In [10]:
formatted_example = format_as_message(example)
print(f"formatted_example: {formatted_example}")

formatted_example: {'id': 'ex_0003', 'category': 'workplace', 'messages': [{'role': 'system', 'content': 'You are a helpful writing assistant. Rewrite messages so they sound natural, clear, and human while preserving the original meaning. Do not add new information.'}, {'role': 'user', 'content': 'Instruction: Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.\n\nInput: Please be informed that I have attached the required document for your reference.\n\n'}, {'role': 'assistant', 'content': 'I’ve attached the required document for your reference.'}]}


In [11]:
for message in formatted_example["messages"]:
    print("=" * 80)
    print("ROLE:", message["role"])
    print(message["content"])

ROLE: system
You are a helpful writing assistant. Rewrite messages so they sound natural, clear, and human while preserving the original meaning. Do not add new information.
ROLE: user
Instruction: Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.

Input: Please be informed that I have attached the required document for your reference.


ROLE: assistant
I’ve attached the required document for your reference.


In [12]:
formatted_preview = [format_as_message(row) for row in train_rows[:5]]

len(formatted_preview)

5

In [13]:
formatted_preview[0]

{'id': 'ex_0003',
 'category': 'workplace',
 'messages': [{'role': 'system',
   'content': 'You are a helpful writing assistant. Rewrite messages so they sound natural, clear, and human while preserving the original meaning. Do not add new information.'},
  {'role': 'user',
   'content': 'Instruction: Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.\n\nInput: Please be informed that I have attached the required document for your reference.\n\n'},
  {'role': 'assistant',
   'content': 'I’ve attached the required document for your reference.'}]}

Conversation Format

We converted raw instruction/input/output rows into a conversation format.

Each formatted example contains:

- a system message that defines the assistant's behavior
- a user message that includes the rewrite instruction and original message
- an assistant message that contains the target rewritten output

This format is useful because instruction-tuned models are already trained to follow chat-style prompts.